# COMP6713 — Notebook 1: Data Exploration

This notebook explores the three medical QA datasets used in this project:
- **PubMedQA** (1,000 labelled pairs) — biomedical yes/no/maybe QA from PubMed abstracts  
- **BioASQ** (8,216 pairs) — factoid biomedical QA  
- **MedQA-USMLE** (11,451 pairs) — US medical licensing exam questions  

All datasets load directly from HuggingFace — no account required.

In [1]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')
import pandas as pd
from collections import Counter
from medqa.data.loader import load_pubmedqa, load_bioasq, load_medqa_usmle, load_all

## 1. Load Datasets

In [2]:
pubmedqa = load_pubmedqa()
bioasq   = load_bioasq()
usmle    = load_medqa_usmle()

print(f"PubMedQA : {len(pubmedqa):>6} records")
print(f"BioASQ   : {len(bioasq):>6} records")
print(f"USMLE    : {len(usmle):>6} records")
print(f"Total    : {len(pubmedqa)+len(bioasq)+len(usmle):>6} records")

[medqa.loader] PubMedQA: loaded 1000 records.
[medqa.loader] BioASQ: loaded 8216 records.
[medqa.loader] MedQA-USMLE: loaded 11451 records.


PubMedQA :   1000 records
BioASQ   :   8216 records
USMLE    :  11451 records
Total    :  20667 records


## 2. Sample Records

In [3]:
# PubMedQA sample
r = pubmedqa[0]
print("=== PubMedQA Sample ===")
print(f"Question : {r['question']}")
print(f"Answer   : {r['answer'][:200]}")
print(f"Type     : {r['answer_type']}")
print(f"Context  : {r['context'][:300]}...")

=== PubMedQA Sample ===
Question : Do mitochondria play a role in remodelling lace plant leaves during programmed cell death?
Answer   : Results depicted mitochondrial dynamics in vivo as PCD progresses within the lace plant, and highlight the correlation of this organelle with other organelles during developmental PCD. To the best of 
Type     : yes
Context  : Programmed cell death (PCD) is the regulated death of cells within an organism. The lace plant (Aponogeton madagascariensis) produces perforations in its leaves through PCD. The leaves of the plant consist of a latticework of longitudinal and transverse veins enclosing areoles. PCD occurs in the cel...


In [4]:
# BioASQ sample
r = bioasq[0]
print("=== BioASQ Sample ===")
print(f"Question : {r['question']}")
print(f"Answer   : {r['answer']}")
print(f"Context  : {r['context'][:300]}...")

=== BioASQ Sample ===
Question : What is the inheritance pattern of Li–Fraumeni syndrome?
Answer   : autosomal dominant
Context  : Balanced t(11;15)(q23;q15) in a TP53+/+ breast cancer patient from a Li-Fraumeni syndrome family. Li-Fraumeni Syndrome (LFS) is characterized by early-onset carcinogenesis involving multiple tumor types and shows autosomal dominant inheritance. Approximately 70% of LFS cases are due to germline muta...


In [5]:
# MedQA-USMLE sample
r = usmle[0]
print("=== MedQA-USMLE Sample ===")
print(f"Question : {r['question']}")
print(f"Options  : {r['context']}")
print(f"Answer   : {r['answer']}")

=== MedQA-USMLE Sample ===
Question : A 23-year-old pregnant woman at 22 weeks gestation presents with burning upon urination. She states it started 1 day ago and has been worsening despite drinking more water and taking cranberry extract. She otherwise feels well and is followed by a doctor for her pregnancy. Her temperature is 97.7°F (36.5°C), blood pressure is 122/77 mmHg, pulse is 80/min, respirations are 19/min, and oxygen saturation is 98% on room air. Physical exam is notable for an absence of costovertebral angle tenderness and a gravid uterus. Which of the following is the best treatment for this patient?
Options  : A: Ampicillin | B: Ceftriaxone | C: Doxycycline | D: Nitrofurantoin
Answer   : Nitrofurantoin


## 3. Dataset Statistics

In [6]:
all_records = load_all()
df = pd.DataFrame(all_records)

print("Dataset distribution:")
print(df['source'].value_counts())
print(f"\nTotal labelled records: {len(df)}")

[medqa.loader] PubMedQA: loaded 1000 records.
[medqa.loader] BioASQ: loaded 8216 records.
[medqa.loader] MedQA-USMLE: loaded 11451 records.


Dataset distribution:
source
medqa_usmle    11451
bioasq          8216
pubmedqa        1000
Name: count, dtype: int64

Total labelled records: 20667


In [7]:
# Question length distribution
df['q_len'] = df['question'].str.len()
df['ctx_len'] = df['context'].str.len()
df['ans_len'] = df['answer'].str.len()

print("Question length (chars):")
print(df.groupby('source')['q_len'].describe()[['mean','min','max']].round(1))
print("\nContext length (chars):")
print(df.groupby('source')['ctx_len'].describe()[['mean','min','max']].round(1))

Question length (chars):
              mean   min     max
source                          
bioasq        63.9  15.0   168.0
medqa_usmle  726.5  66.0  3577.0
pubmedqa      94.2  22.0   213.0

Context length (chars):
               mean    min     max
source                            
bioasq       1603.2  155.0  4374.0
medqa_usmle   128.3   25.0   807.0
pubmedqa     1341.3  293.0  2725.0


In [8]:
# PubMedQA answer type distribution
pqa_df = pd.DataFrame(pubmedqa)
print("PubMedQA answer types:")
print(pqa_df['answer_type'].value_counts())

PubMedQA answer types:
answer_type
yes      552
no       338
maybe    110
Name: count, dtype: int64


## 4. Train / Test Split

In [9]:
from medqa.data.preprocessor import split_dataset
train, test = split_dataset(all_records)
print(f"Train set : {len(train)} records ({len(train)/len(all_records)*100:.1f}%)")
print(f"Test set  : {len(test)} records ({len(test)/len(all_records)*100:.1f}%)")

[medqa.preprocessor] Split -> train: 16533, test: 4134 (stratified=True)


Train set : 16533 records (80.0%)
Test set  : 4134 records (20.0%)


## Summary

| Dataset | Records | Type | Source |
|---------|---------|------|--------|
| PubMedQA | 1,000 | Yes/No/Maybe QA | PubMed abstracts |
| BioASQ | 8,216 | Factoid QA | Biomedical literature |
| MedQA-USMLE | 11,451 | Multiple choice | US medical licensing exam |
| **Total** | **20,667** | | |

The combined dataset of ~20k records is used for training and evaluation.
An additional ~61k unlabelled PubMedQA records are used to enrich the RAG vector store.